[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-3-cleaning-and-exploring/notebook-3.2-fixing-data-types.ipynb)


# Module 3 — Fixing Data Types (Colab walkthrough)

A hands-on companion to the lecture on **fixing data types**, run on the same small, **synthetic** Barcelona-Bicing-style table of per-station snapshots.

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: spotting when a column's *type* quietly hands you a wrong answer, converting it to the right type, and confirming the answer is now correct. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a result you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The first code cell **generates the CSV** (seeded, so results are identical every run) into the current folder — no upload needed.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this data is generated below.


In [1]:
import os
import numpy as np
import pandas as pd

# Where the CSV is written/read. Default "." works in Colab (/content).
DATA_DIR = os.environ.get("MODULE3_DATA_DIR", ".")
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)


DATA_DIR = /private/tmp/claude-501/-Users-pedromarcelino-Documents-xval-bmad-ed-courses-mobillity-univ/3635fdd0-7a4e-4381-b73d-6946be2a9b5e/scratchpad/exec_dir


## Section 0 — Generate the dataset

This cell creates the CSV used below. The generator is seeded
(`np.random.default_rng(42)`), so every run produces the exact same data.


In [2]:
import numpy as np
import pandas as pd

SEED = 42

BCN_STREETS = [
    "Gran Via Corts Catalanes", "Roger de Flor", "Napols", "Carrer de Arago",
    "Passeig de Gracia", "Carrer de Balmes", "Rambla Catalunya", "Carrer de Muntaner",
    "Avinguda Diagonal", "Carrer de Sardenya", "Carrer de Provenca", "Carrer de Valencia",
    "Carrer de Mallorca", "Passeig de Sant Joan", "Carrer de Pau Claris", "Via Laietana",
    "Carrer de Tarragona", "Carrer de Sants", "Avinguda Meridiana", "Carrer del Rossello",
]

def build_snapshots(rng):
    """Barcelona Bicing per-station snapshots (synthetic) — stories 3.1 & 3.2."""
    n_stations = 52
    snaps_per = 24
    n = n_stations * snaps_per  # 1248
    ids = np.repeat(np.arange(1, n_stations + 1), snaps_per)

    st_lat = 41.36 + rng.random(n_stations) * 0.10
    st_lon = 2.12 + rng.random(n_stations) * 0.08
    st_alt = rng.integers(5, 120, n_stations)
    st_slots = rng.integers(18, 40, n_stations)
    st_street = rng.choice(BCN_STREETS, n_stations)

    latitude = np.round(st_lat[ids - 1], 6).astype(object)
    longitude = np.round(st_lon[ids - 1], 6).astype(object)
    altitude = st_alt[ids - 1]
    slots = st_slots[ids - 1]
    streetName = st_street[ids - 1]

    type_col = np.where(rng.random(n) < 0.18, "ELECTRIC_BIKE", "BIKE")

    bikes = np.array([rng.integers(0, s + 1) for s in slots])
    glitch_idx = rng.choice(n, size=6, replace=False)         # a few bikes>slots glitches
    bikes[glitch_idx] = slots[glitch_idx] + rng.integers(1, 4, size=glitch_idx.size)

    # streetNumber: stored as TEXT, mostly numeric, with non-numeric tokens
    streetNumber = rng.integers(1, 320, n).astype(str).astype(object)
    sn_idx = rng.choice(n, size=30, replace=False)
    streetNumber[sn_idx] = "s/n"                              # "sin numero"
    rest = np.setdiff1d(np.arange(n), sn_idx)
    snn_idx = rng.choice(rest, size=4, replace=False)
    streetNumber[snn_idx] = "SN"
    rest2 = np.setdiff1d(rest, snn_idx)
    rng_idx = rng.choice(rest2, size=3, replace=False)
    streetNumber[rng_idx] = "23-25"                           # a range, not a number

    # nearbyStations: comma-separated list stored as TEXT, some blank
    nearby = np.empty(n, dtype=object)
    for i in range(n):
        k = rng.integers(2, 5)
        others = rng.choice(np.arange(1, n_stations + 1), size=k, replace=False)
        nearby[i] = ",".join(str(x) for x in others)
    nearby[rng.choice(n, size=22, replace=False)] = ""

    # station status: every row is "OPN" (open) in this teaching set. Consume one RNG
    # draw here so the columns generated below do not depend on how status is built.
    _ = rng.random(n)                                         # preserve the RNG stream
    status = np.full(n, "OPN", dtype=object)

    # updateTime: European DD/MM/YY HH:MM:SS TEXT.
    #   day "01" is the plurality -> a naive str[:2] "hour" returns 01 (the day);
    #   true activity peaks at hour 08 -> correct busiest hour is 8.
    days = np.empty(n, dtype=int)
    days[:360] = 1
    days[360:] = rng.integers(2, 19, n - 360)
    rng.shuffle(days)
    hour_choices = np.array([0, 1, 2, 3, 6, 7, 8, 9, 12, 13, 17, 18, 19, 22, 23])
    hour_probs = np.array([1, 1, 1, 1, 2, 4, 9, 5, 3, 3, 4, 5, 3, 2, 1], float)
    hour_probs /= hour_probs.sum()
    hours = rng.choice(hour_choices, size=n, p=hour_probs)
    minutes = rng.integers(0, 60, n)
    seconds = rng.integers(0, 60, n)
    updateTime = np.array(
        [f"{d:02d}/03/19 {h:02d}:{mi:02d}:{se:02d}"
         for d, h, mi, se in zip(days, hours, minutes, seconds)], dtype=object)

    # exactly 17 rows with a missing coordinate -> 1248 drops to 1231
    miss = rng.choice(n, size=17, replace=False)
    for i in np.concatenate([miss[:7], miss[12:]]):
        latitude[i] = ""
    for i in np.concatenate([miss[7:12], miss[12:]]):
        longitude[i] = ""

    return pd.DataFrame({
        "id": ids, "type": type_col, "latitude": latitude, "longitude": longitude,
        "streetName": streetName, "streetNumber": streetNumber, "altitude": altitude,
        "slots": slots, "bikes": bikes, "nearbyStations": nearby,
        "status": status, "updateTime": updateTime,
    })

rng = np.random.default_rng(SEED)
snapshots = build_snapshots(rng)
snapshots.to_csv(f"{DATA_DIR}/station-snapshots-sample.csv", index=False)
print("Generated station-snapshots-sample.csv :", len(snapshots), "rows  (stories 3.1, 3.2)")

Generated station-snapshots-sample.csv : 1248 rows  (stories 3.1, 3.2)


## Fixing Data Types

*The hook:* you ask for the busiest bike-share hour, the code runs with no error, and
returns **01:00**. The cause: a key column is still text. We load normally, **audit**
the dtypes, **convert**, then **verify**.


In [3]:
snap = pd.read_csv(f"{DATA_DIR}/station-snapshots-sample.csv")   # normal load
print("dtypes:")
print(snap.dtypes.to_string())
print()
for c in ["streetNumber", "updateTime", "nearbyStations"]:
    print(f"  {c:15s} -> {str(snap[c].dtype):8s} e.g. {snap[c].dropna().iloc[0]!r}")


dtypes:
id                  int64
type                  str
latitude          float64
longitude         float64
streetName            str
streetNumber          str
altitude            int64
slots               int64
bikes               int64
nearbyStations        str
status                str
updateTime            str

  streetNumber    -> str      e.g. '161'
  updateTime      -> str      e.g. '01/03/19 08:25:41'
  nearbyStations  -> str      e.g. '35,19'


In [4]:
# The silent failure: updateTime is TEXT in 'DD/MM/YY HH:MM:SS' form.
# A naive 'take the first two characters as the hour' actually grabs the DAY.
naive_busiest = snap["updateTime"].str[:2].value_counts().idxmax()
print("NAIVE busiest 'hour' (wrong):", naive_busiest, "  <- this is the day-of-month, not the hour")


NAIVE busiest 'hour' (wrong): 01   <- this is the day-of-month, not the hour


In [5]:
# Convert numeric-looking text; invalid tokens become NaN (errors='coerce').
street_num = pd.to_numeric(snap["streetNumber"], errors="coerce")
became_nan = int(street_num.isna().sum())
print("streetNumber rows -> NaN after to_numeric(coerce):", became_nan)
print("  offending tokens:",
      sorted(snap.loc[street_num.isna(), "streetNumber"].str.strip().unique().tolist()))

# Convert date text with the explicit European format.
update_dt = pd.to_datetime(snap["updateTime"], format="%d/%m/%y %H:%M:%S", errors="coerce")
print("updateTime rows that failed to parse (NaT):", int(update_dt.isna().sum()))

# Now the busiest hour is correct.
correct_busiest = int(update_dt.dt.hour.value_counts().idxmax())
print("CORRECT busiest hour (after fixing the type):", correct_busiest)

# Normalize the list-like text column into real lists.
nearby_lists = snap["nearbyStations"].fillna("").apply(
    lambda s: [x for x in str(s).split(",") if x])
print("nearbyStations normalized, e.g.:", nearby_lists.iloc[0])

print("re-audit -> streetNumber:", street_num.dtype, "| updateTime:", update_dt.dtype)


streetNumber rows -> NaN after to_numeric(coerce): 37
  offending tokens: ['23-25', 'SN', 's/n']
updateTime rows that failed to parse (NaT): 0
CORRECT busiest hour (after fixing the type): 8
nearbyStations normalized, e.g.: ['35', '19']
re-audit -> streetNumber: float64 | updateTime: datetime64[us]


## Done — fixing data types

The data-types lecture has run on the synthetic snapshots: the dtype **audit**, the
silent `01:00` vs the **correct** busiest hour (8) after fixing the type, the 37 invalid
`streetNumber` tokens coerced to NaN, and the list-in-text column normalized. Re-run
top-to-bottom any time; the seeded generator reproduces identical numbers. Then try the
matching exercise yourself.
